In [4]:
# 라벨링 오류 검증
import pandas as pd
import re
import os

# --- 설정 ---
INPUT_FILE = 'labeled_data_only.csv'
LABEL_COLUMN = 'label(chatgpt,claude,grok,gemini)'

def find_error_rows(filepath) :
    """
    CSV 파일을 읽어 라벨 형식에 오류가 있는 행의 번호와 내용을 출력합니다.
    """
    print(f"데이터 오류 검사를 시작합니다. 입력 파일: '{filepath}'")

    try :
        df = pd.read_csv(filepath)
        print(f"총 {len(df)}개의 데이터를 읽었습니다.")
    except FileNotFoundError :
        print(f"\n[오류] 파일을 찾을 수 없습니다: '{filepath}'")
        print("스크립트와 같은 폴더에 파일이 있는지 확인해주세요.")
        return

    error_details = []

    # 모든 행을 순회하며 라벨 유효성 검사
    for index, row in df.iterrows() :
        label_string = row[LABEL_COLUMN]
        is_valid = False
        error_reason = "알 수 없는 오류"  # 기본 오류 메시지

        if isinstance(label_string, str) :
            try :
                # 정규식을 사용해 숫자만 추출
                found_numbers = re.findall(r'-?\d+', label_string)

                # 규칙 1: 숫자가 정확히 4개여야 함
                if len(found_numbers) != 4 :
                    error_reason = f"라벨 개수가 4개가 아님 (발견된 개수: {len(found_numbers)})"
                else :
                    # 규칙 2: 숫자를 정수로 변환
                    parsed_list = [int(num) for num in found_numbers]

                    # 규칙 3: 모든 숫자가 -1, 0, 1 중 하나여야 함
                    if not all(num in [-1, 0, 1] for num in parsed_list) :
                        error_reason = f"허용되지 않은 숫자 포함: {parsed_list}"
                    else :
                        is_valid = True
            except :
                error_reason = "숫자 변환 중 오류 발생"
        else :
            error_reason = "라벨이 문자열 형식이 아니거나 비어있음"

        # 유효하지 않은 경우, 오류 목록에 추가
        if not is_valid :
            error_details.append({
                'row_number' : index + 2,  # 실제 파일 행 번호 (헤더 포함)
                'original_label' : label_string,
                'reason' : error_reason
            })

    # --- 최종 결과 보고 ---
    print("\n--- 검사 결과 ---")
    if not error_details :
        print("🎉 모든 데이터의 라벨 형식이 유효합니다!")
    else :
        print(f"❌ 총 {len(error_details)}개의 형식 오류 데이터를 발견했습니다.")
        print("아래 목록을 참고하여 원본 파일을 직접 수정해주세요.\n")

        # 발견된 모든 오류를 출력
        for error in error_details :
            print(f"▶ 파일 행 번호: {error['row_number']}")
            print(f"  원본 라벨: '{error['original_label']}'")
            print(f"  오류 원인: {error['reason']}")
            print("-" * 30)

    print("\n데이터 오류 검사가 완료되었습니다.")


if __name__ == '__main__' :
    find_error_rows(INPUT_FILE)


데이터 오류 검사를 시작합니다. 입력 파일: 'labeled_data_only.csv'
총 1305개의 데이터를 읽었습니다.

--- 검사 결과 ---
🎉 모든 데이터의 라벨 형식이 유효합니다!

데이터 오류 검사가 완료되었습니다.


In [ ]:
#라벨 형식 통일
import pandas as pd
import ast
import re

def normalize_label_format(csv_path, output_path=None):
    """
    labeled_data.csv 파일의 라벨 형식을 통일합니다 (공백 제거).

    Args:
        csv_path (str): 입력 CSV 파일 경로
        output_path (str): 출력 CSV 파일 경로 (None이면 원본 파일을 덮어씀)

    Returns:
        int: 변경된 행의 수
    """

    # CSV 파일 읽기
    df = pd.read_csv(csv_path)

    # 라벨 컬럼명
    label_column = "label(chatgpt,claude,grok,gemini)"

    changed_count = 0

    for idx, row in df.iterrows():
        try:
            label_str = str(row[label_column]).strip()

            # 이미 올바른 형식인지 확인
            if label_str.startswith('[') and label_str.endswith(']'):
                # 리스트로 파싱해서 검증
                try:
                    labels = ast.literal_eval(label_str)
                    if isinstance(labels, list) and len(labels) == 4:
                        # 공백 없는 형식으로 통일
                        normalized = str(labels).replace(' ', '')

                        # 변경이 필요한 경우
                        if label_str != normalized:
                            df.loc[idx, label_column] = normalized
                            changed_count += 1

                except (ValueError, SyntaxError):
                    # 파싱할 수 없는 경우는 그대로 두기
                    continue

        except Exception as e:
            # 예외 발생 시 해당 행은 그대로 두기
            continue

    # 파일 저장
    if output_path is None:
        output_path = csv_path

    df.to_csv(output_path, index=False)

    return changed_count

def preview_label_changes(csv_path, num_samples=10):
    """
    라벨 형식 변경 전후를 미리보기합니다.

    Args:
        csv_path (str): CSV 파일 경로
        num_samples (int): 보여줄 샘플 수
    """

    df = pd.read_csv(csv_path)
    label_column = "label(chatgpt,claude,grok,gemini)"

    print("라벨 형식 변경 미리보기:")
    print("=" * 80)

    changes_found = 0

    for idx, row in df.iterrows():
        if changes_found >= num_samples:
            break

        try:
            label_str = str(row[label_column]).strip()

            if label_str.startswith('[') and label_str.endswith(']'):
                try:
                    labels = ast.literal_eval(label_str)
                    if isinstance(labels, list) and len(labels) == 4:
                        normalized = str(labels).replace(' ', '')

                        if label_str != normalized:
                            print(f"행 {idx}:")
                            print(f"  변경 전: {label_str}")
                            print(f"  변경 후: {normalized}")
                            print()
                            changes_found += 1

                except (ValueError, SyntaxError):
                    continue

        except Exception:
            continue

    if changes_found == 0:
        print("변경이 필요한 라벨이 없습니다.")
    else:
        print(f"총 {changes_found}개의 샘플을 보여줍니다.")

def count_format_types(csv_path):
    """
    라벨 형식의 종류를 분석합니다.
    """

    df = pd.read_csv(csv_path)
    label_column = "label(chatgpt,claude,grok,gemini)"

    with_spaces = 0
    without_spaces = 0
    invalid_format = 0

    for idx, row in df.iterrows():
        try:
            label_str = str(row[label_column]).strip()

            if label_str.startswith('[') and label_str.endswith(']'):
                try:
                    labels = ast.literal_eval(label_str)
                    if isinstance(labels, list) and len(labels) == 4:
                        if ', ' in label_str:
                            with_spaces += 1
                        else:
                            without_spaces += 1
                    else:
                        invalid_format += 1
                except (ValueError, SyntaxError):
                    invalid_format += 1
            else:
                invalid_format += 1

        except Exception:
            invalid_format += 1

    print("라벨 형식 분석:")
    print(f"공백 있는 형식 (예: [1, 0, -1, 1]): {with_spaces:,}개")
    print(f"공백 없는 형식 (예: [1,0,-1,1]): {without_spaces:,}개")
    print(f"잘못된 형식: {invalid_format:,}개")
    print(f"전체: {len(df):,}개")

if __name__ == "__main__":
    csv_path = "../raw_data/junk/labeled_data_1.csv"

    # 형식 분석
    count_format_types(csv_path)
    print()

    # 변경 미리보기
    preview_label_changes(csv_path, 5)
    print()

    # 사용자 확인
    response = input("라벨 형식을 통일하시겠습니까? (y/n): ")

    if response.lower() == 'y':
        # 백업 생성
        backup_path = csv_path.replace('.csv', '_backup.csv')
        print(f"백업 파일 생성: {backup_path}")
        df_backup = pd.read_csv(csv_path)
        df_backup.to_csv(backup_path, index=False)

        # 형식 통일
        changed_count = normalize_label_format(csv_path)
        print(f"완료! {changed_count:,}개의 라벨 형식이 통일되었습니다.")
    else:
        print("작업이 취소되었습니다.")